# 🧠 Language & Cognition — Word Frequency Analysis

In this notebook you'll analyze a real book and discover **Zipf's Law** —
a famous pattern in language that tells us how our brains balance effort and meaning.

**Run each cell in order by pressing `Shift + Enter`.**

---
### What is Zipf's Law?
In almost every text ever written, a tiny group of words (like *the*, *of*, *and*) appears
an enormous number of times, while thousands of other words appear only once or twice.
This is not random — it reflects a deep principle of how human cognition balances
**ease of communication** with **information richness**.

---
## Step 1 — Install & import libraries

Libraries are pre-built tools other people wrote so you don't have to start from scratch.
- **pandas** → works with tables of data
- **matplotlib** → draws charts
- **wordcloud** → draws word clouds
- **re** → helps clean text (built into Python, no install needed)

In [ ]:
# Run this cell first to install the libraries you need
# The exclamation mark means: run this as a terminal command
!pip install pandas matplotlib wordcloud --quiet

In [ ]:
import re                          # for cleaning text
import pandas as pd                # for working with data tables
import matplotlib.pyplot as plt    # for drawing charts
from wordcloud import WordCloud    # for drawing word clouds
from collections import Counter    # for counting things

print('✅ All libraries imported successfully!')

---
## Step 2 — Load the book

Make sure you've downloaded `Alice's Adventures in Wonderland` from Project Gutenberg
and saved it as `data/book.txt`.

Download link: https://www.gutenberg.org/files/11/11-0.txt

👉 Right-click → Save As → save inside the `data/` folder as `book.txt`

In [ ]:
# Open the file and read all the text into a variable called 'text'
with open('data/book.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Let's see how long the book is
print(f'Total characters in the book: {len(text):,}')
print()

# Preview the first 500 characters
print('--- First 500 characters ---')
print(text[:500])

---
## Step 3 — Clean the text

Raw text from the internet is messy — it has punctuation, capital letters,
numbers, and header/footer text from Project Gutenberg.
We need to clean it before counting words.

**Steps:**
1. Convert everything to lowercase (so `The` and `the` count as the same word)
2. Remove punctuation and numbers
3. Split into a list of individual words

In [ ]:
# Step 1: lowercase everything
text_lower = text.lower()

# Step 2: keep only letters and spaces (remove everything else)
# re.sub replaces anything that is NOT a letter or space with nothing ''
text_clean = re.sub(r'[^a-z\s]', '', text_lower)

# Step 3: split the text into a list of words
words = text_clean.split()

print(f'Total words in the book: {len(words):,}')
print(f'Preview of first 20 words: {words[:20]}')

---
## Step 4 — Count word frequencies

Now we count how many times each word appears.
`Counter` takes a list and returns a dictionary of `{word: count}` pairs.

In [ ]:
# Count how often each word appears
word_counts = Counter(words)

# Convert to a pandas DataFrame (like a spreadsheet)
df = pd.DataFrame(word_counts.items(), columns=['word', 'count'])

# Sort by count, highest first
df = df.sort_values('count', ascending=False).reset_index(drop=True)

# Add a 'rank' column (1 = most common)
df['rank'] = df.index + 1

print(f'Unique words found: {len(df):,}')
print()
print('Top 20 most common words:')
print(df.head(20).to_string(index=False))

---
## Step 5 — Remove stopwords

The most common words (*the*, *a*, *of*, *and*) are called **stopwords**.
They're grammatically necessary but tell us nothing about what a text is *about*.

Cognitively, these are the words our brains process almost automatically — 
they're so overlearned they require almost no mental effort.

Let's filter them out to find the *meaningful* words.

In [ ]:
# A basic list of English stopwords
stopwords = set([
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
    'for', 'of', 'with', 'is', 'was', 'are', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'did', 'does', 'it', 'its', 'that',
    'this', 'i', 'he', 'she', 'they', 'we', 'you', 'me', 'him', 'her',
    'them', 'us', 'my', 'his', 'their', 'our', 'your', 'said', 'not',
    'so', 'as', 'if', 'up', 'out', 'no', 'by', 'what', 'all', 'about',
    'would', 'could', 'which', 'who', 'from', 'into', 'there', 'one',
    'will', 'just', 'when', 'very', 'more', 'then', 'than', 'how',
    'go', 'went', 'come', 'came', 'know', 'think', 'can', 'get'
])

# Filter out stopwords
df_content = df[~df['word'].isin(stopwords)].copy()
df_content = df_content.reset_index(drop=True)
df_content['rank'] = df_content.index + 1

print('Top 20 most common CONTENT words (stopwords removed):')
print(df_content.head(20).to_string(index=False))

---
## Step 6 — Visualize the top words (bar chart)

Let's make a bar chart of the top 20 content words.
This is a simple but powerful way to see what a text is "about" at a glance.

In [ ]:
top20 = df_content.head(20)

plt.figure(figsize=(12, 5))
plt.bar(top20['word'], top20['count'], color='steelblue', edgecolor='white')
plt.title('Top 20 content words in Alice in Wonderland', fontsize=14, pad=12)
plt.xlabel('Word')
plt.ylabel('Times it appears')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('top_words_bar.png', dpi=150)
plt.show()
print('Chart saved as top_words_bar.png')

---
## Step 7 — Discover Zipf's Law 📉

Here's where cognitive science meets data science.

**Zipf's Law** predicts: if you plot *rank* vs *frequency* on a log-log scale,
you get a straight line. This holds for almost every human language ever studied.

Why does this matter cognitively?
- Our brains **minimize effort** by reusing a few common words constantly
- At the same time, we need **many rare words** to express specific ideas
- Zipf's Law is the mathematical "sweet spot" between these two pressures

In [ ]:
import numpy as np

# Use all words (including stopwords) for Zipf's Law
# We need enough words to see the full distribution
df_zipf = df.head(1000).copy()

plt.figure(figsize=(10, 6))

# Plot the actual data
plt.loglog(df_zipf['rank'], df_zipf['count'],
           'o', color='steelblue', alpha=0.5, markersize=4, label='Actual word frequencies')

# Plot what Zipf's Law predicts (a perfect straight line on log-log scale)
ranks = np.array(df_zipf['rank'])
zipf_prediction = df_zipf['count'].iloc[0] / ranks   # frequency ∝ 1/rank
plt.loglog(ranks, zipf_prediction,
           '--', color='coral', linewidth=2, label="Zipf's Law prediction")

plt.title("Zipf's Law in Alice in Wonderland", fontsize=14, pad=12)
plt.xlabel('Rank (1 = most common word)')
plt.ylabel('Frequency (times it appears)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('zipf_law.png', dpi=150)
plt.show()

print("Notice how closely the real data (blue dots) follows Zipf's prediction (dashed line)!")
print("This pattern appears in almost every human language.")

---
## Step 8 — Word cloud 🌥️

A word cloud shows word importance visually — bigger words appear more often.
It's a quick way to get an intuition for what a text is about.

In [ ]:
# Build a frequency dictionary from content words only
freq_dict = dict(zip(df_content['word'], df_content['count']))

# Generate the word cloud
wc = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='Blues',
    max_words=100
).generate_from_frequencies(freq_dict)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most common content words in Alice in Wonderland', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150)
plt.show()
print('Word cloud saved as wordcloud.png')

---
## Step 9 — Reflect on what you found

Answer these questions in the cell below by replacing the `...` with your answers.
This is the cognitive science part — thinking about what the data means!

### 📝 Your reflections

**1. What are the top 5 content words in the book? Does that match what you know about the story?**

*Your answer: ...*

**2. Does the Zipf's Law chart match the prediction well? Where does it deviate?**

*Your answer: ...*

**3. What surprised you about the word frequencies?**

*Your answer: ...*

**4. If you were the author, what words would you expect to be common? Were you right?**

*Your answer: ...*

---
## 🎯 Bonus challenges

If you want to go further, try these on your own:

1. **Change the book** — Download a different book from [Project Gutenberg](https://www.gutenberg.org/) and re-run. Does Zipf's Law still hold?

2. **Compare two books** — Load two books, count words in each, and plot them side-by-side.
   Which author uses a wider vocabulary?

3. **Sentence length** — Split the text by sentences instead of words. Plot a histogram of sentence lengths.
   Does the author tend to write short or long sentences?

4. **Vocabulary richness** — Calculate the *type-token ratio*: unique words ÷ total words.
   A higher ratio means the author uses more varied vocabulary. Try: `len(df) / len(words)`

In [ ]:
# 💡 Scratch space — try your bonus challenges here!

# Vocabulary richness (type-token ratio)
ratio = len(df) / len(words)
print(f'Vocabulary richness (type-token ratio): {ratio:.4f}')
print(f'This means for every 10,000 words, {int(ratio*10000)} are unique.')

---
## ✅ Summary — what you practiced

| Skill | Where you used it |
|---|---|
| Reading files | Step 2 — loading the book |
| String cleaning | Step 3 — lowercase, remove punctuation |
| Counting with Python | Step 4 — `Counter` |
| pandas DataFrames | Step 4 & 5 — sorting, filtering |
| Bar charts | Step 6 — `matplotlib` |
| Log-log plots | Step 7 — Zipf's Law |
| Word clouds | Step 8 — `WordCloud` |
| Critical thinking | Step 9 — reflections |

**Great work! Commit your notebook to GitHub and check off this project in your README.**